# 行业配置策略：拥挤度视角 - 研报复现

**基于华泰证券研报《基本面轮动系列之九：行业配置策略-拥挤度视角》**

本notebook复现研报中的核心策略：
1. 拥挤度指标构建与验证
2. 月度空头行业轮动策略
3. 日度行业风险监控策略
4. 大盘择时策略
5. 景气度+拥挤度复合策略

In [ ]:
# 设置路径和导入模块
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from source.data_fetcher import (
    get_sw_industry_daily, load_cached_data, get_all_industries_data,
    get_market_index_data, pro
)
from source.crowdedness import CrowdednessIndicator, calculate_forward_returns
from source.rotation_strategy import IndustryRotationStrategy, ProsperityCrowdednessStrategy
from source.backtest import run_backtest, compare_strategies, PerformanceAnalyzer
from source.risk_management import RiskManager, CrowdMonitor
from source.visualization import (
    plot_equity_curves, plot_drawdown_series, plot_strategy_comparison,
    create_performance_summary, plot_crowdedness_heatmap
)

print('模块导入成功！')

## 1. 数据获取

In [ ]:
# 尝试加载缓存数据，如果不存在则重新获取
industry_data = load_cached_data()

if industry_data is None:
    print('缓存数据不存在，正在从tushare获取...')
    industry_data = get_all_industries_data(
        start_date='20180101',  # 为了演示，使用较短的时间范围
        end_date='20231231'
    )
else:
    print(f'成功加载缓存数据，共 {len(industry_data)} 个行业')

In [ ]:
# 获取市场基准数据 (上证指数)
benchmark = get_market_index_data('000001.SH', '20180101', '20231231')
print(f'基准数据获取成功，共 {len(benchmark)} 条记录')
benchmark_returns = benchmark['close'].pct_change().dropna()

## 2. 拥挤度指标计算

In [ ]:
# 初始化拥挤度指标计算器
indicator = CrowdednessIndicator()

# 计算各行业的拥挤度指标
all_crowdedness = {}
all_forward_returns = {}

for code, data in list(industry_data.items())[:10]:  # 先用10个行业演示
    try:
        # 计算所有指标
        indicators = indicator.calculate_all_indicators(data)
        
        # 计算复合拥挤度
        composite = indicator.calculate_composite_crowdedness(indicators)
        
        # 计算未来20日收益率
        forward_ret = calculate_forward_returns(data, [20])[20]
        
        all_crowdedness[code] = composite['composite_crowdedness']
        all_forward_returns[code] = forward_ret
        
        print(f'{code}: 拥挤度计算完成')
    except Exception as e:
        print(f'{code}: 计算失败 - {e}')
        continue

# 合并为DataFrame
crowdedness_signals = pd.DataFrame(all_crowdedness)
print(f'\n拥挤度信号计算完成，形状: {crowdedness_signals.shape}')

## 3. 策略一：月度空头行业轮动策略

In [ ]:
# 运行月度空头策略
rotation = IndustryRotationStrategy(industry_data, crowdedness_signals)
rotation.calculate_benchmark_returns()

print('运行策略一：月度空头行业轮动...')
strat1_returns, strat1_pos = rotation.strategy_one_monthly_short()

# 回测
results1 = run_backtest(
    strat1_returns.dropna(),
    rotation.benchmark_returns,
    initial_capital=1000000,
    strategy_name='月度空头轮动'
)

## 4. 策略二：日度行业风险监控策略

In [ ]:
print('运行策略二：日度行业风险监控...')
strat2_returns, strat2_portfolio = rotation.strategy_two_daily_risk_monitor()

# 回测
results2 = run_backtest(
    strat2_returns.dropna(),
    rotation.benchmark_returns,
    initial_capital=1000000,
    strategy_name='日度风险监控'
)

## 5. 策略三：大盘择时策略

In [ ]:
print('运行策略三：大盘择时...')
strat3_returns, strat3_portfolio = rotation.strategy_three_market_timing(crowded_threshold=10)

# 回测
results3 = run_backtest(
    strat3_returns.dropna(),
    rotation.benchmark_returns,
    initial_capital=1000000,
    strategy_name='大盘择时'
)

## 6. 景气度+拥挤度复合策略

In [ ]:
print('运行景气度+拥挤度复合策略...')
复合 = ProsperityCrowdednessStrategy(industry_data, crowdedness_signals)
comp_returns, comp_excess, comp_signals = 复合.strategy_monthly_prosperity_crowdedness()

# 回测
results4 = run_backtest(
    comp_returns.dropna(),
    复合.calculate_benchmark_returns(),
    initial_capital=1000000,
    strategy_name='景气拥挤复合'
)

## 7. 策略对比分析

In [ ]:
# 汇总所有策略结果
all_results = {
    '月度空头轮动': results1,
    '日度风险监控': results2,
    '大盘择时': results3,
    '景气拥挤复合': results4
}

# 生成对比表格
comparison_df = create_performance_summary(all_results)
print('\n策略表现对比：')
print(comparison_df.to_string(index=False))

In [ ]:
# 绘制策略对比图
plot_strategy_comparison(all_results, save_path='output/strategy_comparison.png')

## 8. 可视化分析

In [ ]:
# 绘制净值曲线
equity_curves = {
    '月度空头轮动': results1['equity_curve'],
    '日度风险监控': results2['equity_curve'],
    '大盘择时': results3['equity_curve'],
    '景气拥挤复合': results4['equity_curve'],
}
plot_equity_curves(equity_curves, title='各策略净值曲线对比', save_path='output/equity_curves.png')

In [ ]:
# 绘制回撤曲线
drawdown_curves = {
    '月度空头轮动': results1['drawdown'],
    '日度风险监控': results2['drawdown'],
    '大盘择时': results3['drawdown'],
    '景气拥挤复合': results4['drawdown'],
}
plot_drawdown_series(drawdown_curves, title='各策略回撤对比', save_path='output/drawdown_curves.png')

## 9. 拥挤度热力图

In [ ]:
# 绘制拥挤度热力图
try:
    plot_crowdedness_heatmap(
        crowdedness_signals,
        title='行业拥挤度时序热力图',
        save_path='output/crowdedness_heatmap.png'
    )
except Exception as e:
    print(f'热力图绘制失败: {e}')

## 10. 结论与说明

**重要提示：**

1. 本项目复现了研报《行业配置策略：拥挤度视角》的核心策略框架

2. 由于缺少研报中使用的完整历史数据和景气度指标原始数据，部分结果可能与研报存在差异

3. 研报中提到的景气度指标来自前期报告《行业配置策略：景气度视角》(2020-11-05)，本项目中使用了简化的景气度代理指标

4. 拥挤度指标的核心逻辑已实现：
   - comp_turn_kurtosis_10: 成分股10日收益率峰度 (阈值95%)
   - turn_20: 过去20日平均换手率 (阈值90%)
   - corr_amount_close_40: 40日成交额收盘价相关系数 (阈值95%)

5. 策略表现受市场环境、数据质量等因素影响，历史表现仅供参考